# 01 · Задача и данные

Модель — ассистент студента, который пишет выпускную работу в редакторе. Рядом с диалогом
открыт фрагмент документа, студент о чём-то просит, модель отвечает одним сообщением.

Никаких инструментов и циклов здесь нет намеренно. Мы работаем над тем, *как* модель пишет
и *где* проводит границы, потому что именно это переносится в любую агентскую обвязку, которую
можно навесить позже.

Чего мы хотим от модели:

- содержательные решения оставлять студенту — тему, цель, гипотезу, выводы;
- опираться только на открытый фрагмент и не выдумывать его содержимое;
- правки формы предлагать к принятию, а не молча переписывать;
- отказывать, когда просят обойти проверку, выдумать данные или написать работу за студента;
- заканчивать одним шагом или одним вопросом, а не анкетой.

Четыре набора: `train` для обучения, `dev` для замера по ходу работы, `test_product` — голд-сет
продукта с рубриками и ответами действующего агента, `test_extended` — наш расширенный тест
на новых предметных областях. Документы обучения и тестов не пересекаются.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import data, metrics, report

splits = {name: data.load(name) for name in data.SPLITS}
for name, ds in splits.items():
    print(f"{name:14} {len(ds):4} ситуаций")
train = splits["train"]

## Формат: как учат реальные модели

Строка — это диалог в формате TRL для обучения на предпочтениях: `prompt` — список сообщений
с ролями, заканчивается репликой студента; `chosen` и `rejected` — по одному сообщению
ассистента. Системный промпт и фрагмент документа уже внутри `prompt`, файл самодостаточен.

Один и тот же файл кормит все методы:

| метод | что берёт из строки |
|---|---|
| SFT | `prompt` + `chosen`, столбец переименовывается в `completion` |
| DPO, ORPO, SimPO | `prompt`, `chosen`, `rejected` как есть |
| KTO | те же строки, распаренные: два примера с меткой хороший / плохой |
| вектор управления | активации на `chosen` против `rejected` |

In [ ]:
row = next(r for r in train if len(r["prompt"]) > 2)
print("роли в prompt:", [m["role"] for m in row["prompt"]])
print("chosen:", row["chosen"])
print()
print("SYSTEM:")
print(row["prompt"][0]["content"])

In [ ]:
data.show(row)

## Что именно тюним: эталон против плохого ответа

Плохой ответ — не мусор. Это грамотный текст, который нарушает ровно одно правило продукта.
Разница между двумя столбцами и есть то, чему учим. Ниже три пары; под каждой — правило,
которое нарушает правый столбец.

In [ ]:
def side_by_side(row, width=60):
    """Reference and bad answer next to each other, one paragraph per line."""
    left, right = row["chosen"][0]["content"].split("\n"), row["rejected"][0]["content"].split("\n")
    print(f"{row['id']} · {data.request(row)[:90]}")
    print(f"{'ЭТАЛОН':<{width}} │ ПЛОХОЙ ОТВЕТ")
    print("─" * (2 * width + 3))
    for i in range(max(len(left), len(right))):
        l = left[i] if i < len(left) else ""
        r = right[i] if i < len(right) else ""
        for j in range(0, max(len(l), len(r), 1), width):
            print(f"{l[j:j + width]:<{width}} │ {r[j:j + width]}")
    print()


for rid in ("TR-TIB-02", "TR-AWR-22", "TR-FT-06"):
    match = [r for r in train if r["id"] == rid]
    if match:
        side_by_side(match[0])

## Автопроверки

Часть требований проверяется механически: заканчивается ли ответ шагом, сколько в нём вопросов,
нет ли готового текста под вставку, опирается ли он на документ. Это дёшево, детерминированно
и не требует модели.

Проверка полезна, только если она проходит на эталоне и не проходит на плохом ответе.
Разделяющая способность — разница этих долей:

$$
\Delta = \frac{1}{N}\sum_i \mathbb{1}[\text{check}(y^+_i)] \;-\; \frac{1}{N}\sum_i \mathbb{1}[\text{check}(y^-_i)]
$$

Проверки с $\Delta \approx 0$ — ограничения, а не измерители: их выполняет любой аккуратный ответ.

In [ ]:
rows = [r for ds in splits.values() for r in ds]
passed = {}
for r in rows:
    case = data.case(r)
    for name, ok in metrics.run(r["chosen"][0]["content"], case).items():
        passed.setdefault(name, [[], []])[0].append(ok)
    for name, ok in metrics.run(r["rejected"][0]["content"], case).items():
        passed[name][1].append(ok)

print(f"{'проверка':16}{'назначено':>10}{'эталон':>9}{'плохой':>9}{'Δ':>8}")
for name, (good, bad) in sorted(passed.items(), key=lambda kv: -(sum(kv[1][0]) / len(kv[1][0]) - sum(kv[1][1]) / len(kv[1][1]))):
    g, b = sum(good) / len(good), sum(bad) / len(bad)
    print(f"{name:16}{len(good):>10}{g:>9.0%}{b:>9.0%}{g - b:>8.0%}")

Содержательные нарушения — «сформулировал гипотезу за студента» — регулярным выражением не
поймать. Для них есть LLM-судья, он появится в следующем ноутбуке вместе с проверкой, насколько
его вердикты совпадают с человеческими.

## Состав наборов

In [ ]:
from collections import Counter

for name, ds in splits.items():
    n = len(ds)
    print(f"{name:14} {n:4}  отказ обязателен {sum(ds['must_refuse']):3}"
          f"  многоходовых {sum(len(p) > 2 for p in ds['prompt']):3}"
          f"  без документа {sum(not d for d in ds['document']):3}"
          f"  длина эталона {sum(len(c[0]['content']) for c in ds['chosen']) // n:4}")

print()
for name in ("train", "test_extended"):
    print(f"{name}:", dict(Counter(splits[name]["category"]).most_common()))

Ситуации с обязательным отказом держатся в меньшинстве намеренно, и рядом с ними лежат
легитимные просьбы, которые выглядят похоже: перефразировать свой текст, проверить свой расчёт,
разобрать отчёт антиплагиата. Если учить отказам вперемешку с обычной работой, модель начинает
отказывать на нормальных просьбах — это главный риск всего обучения, и за ним следит
отдельная метрика.

## Два теста

**Тест продукта** — 33 ситуации от команды продукта: рубрика PASS/FAIL и ответ действующего
агента с вердиктом человека. Три ситуации, которые сама команда пометила как невалидные,
исключены. Это правда продукта, но все ситуации построены на одной работе, и на 33 строках
доверительный интервал шириной около 15 пунктов.

**Расширенный тест** — 100 наших ситуаций на десяти новых документах из других областей:
филология, финансы, биология, логистика, сестринское дело, музыкальная педагогика, информатика.
Кроме шести категорий продукта в нём есть оси, которых в тесте продукта нет: ловушки
на ложный отказ, пустой документ, второй ход диалога, ошибки в статистике, вопросы вне области.

In [ ]:
product = splits["test_product"]
sample = product[0]
print(sample["id"], "·", sample["category"])
print("ЗАПРОС:", data.request(sample))
print("РУБРИКА:")
for item in sample["rubric"]:
    print(" -", item)
print()
print("ОТВЕТ ПРОДОВОГО АГЕНТА:")
print(sample["reference"]["answer"][:700])
print()
print("вердикт человека:", sample["reference"]["human"] or "не заполнен")
print()
print("вердикты человека по всему тесту:", dict(Counter(r["reference"]["human"] or "не заполнен" for r in product)))

In [ ]:
data.show(splits["test_extended"][13])